# Animated entropy-neuron identification plane

Part of the experiments_anim split (1: boundary mean spectra, 2: eigval spectra, 3: coupling heatmaps, 4: layerwise RankMe, 5: layerwise mean norm, 6: entropy-neuron plane) — split so saved outputs stay small enough for the editor.

Left panel: the identification plane of Stolfo et al. (arXiv:2406.16254) for the last block — LogitVar against $\rho$ (the share of a neuron's write direction inside the span of the bottom-$k$ right singular vectors of $W_U$, $k=0.01\,d$), point size = weight norm. One frame per weight checkpoint; axes pinned over the run, so the cloud moves and the box doesn't. Entropy neurons are the bottom-right corner.

Right panel: RankMe and $\alpha$ of the final stream (`after_final_norm`, centered) over the same run, with a red line scanning to the frame's token count — the plane's motion read against the spectrum-entropy curves.

Data: `data/results/entropy_neurons.pt` (weights only, log checkpoint schedule; written by `oneoff_scripts/entropy_neurons.py`). The static version of the plane is section 3 of `analysis/entropy_neurons.py`.

In [ ]:
import os, sys
import shutil
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
import torch
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    get_ys, get_xs_tokens, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, get_model_label, YVAR_LABELS, XVAR_FNS)
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Config this notebook needs (kept out of the generic lib):
NEURONS = 'data/results/entropy_neurons.pt'   # W_U centred over the vocabulary
RANKME_HOOK = 'after_final_norm'              # the stream the unembedding actually reads
FRAC = 0.01                                   # the paper's k = 0.01 d_model

data = torch.load(NEURONS, weights_only=False)
MODELS = [m for m in data if data[m]]
COLORS = dict(zip(MODELS, plt.cm.tab10.colors))
SAMPLES_SRC = {m: 'nanochat_samples' if m == 'nanochat-d12' else 'block_representations_samples'
               for m in MODELS}
print({m: {b: len(v) for b, v in data[m].items()} for m in MODELS})

In [ ]:
# Same helpers as analysis/entropy_neurons.py, kept local so this notebook stands alone.
def last_block(model):
    return sorted(data[model], reverse=True)[0]

def k_index(entry, frac=FRAC):
    """Index into the stored ks of the paper's k ~ frac * d_model."""
    return entry['ks'].index(max(1, round(frac * entry['d_model'])))

def live_steps(per_step):
    """Checkpoints with a non-zero write matrix (nanochat zero-inits c_proj: steps 0-1 give
    rho = 0/0)."""
    return [s for s in sorted(per_step) if float(per_step[s]['norm'].max()) > 0]

def stream_curve(model, yvar):
    """(tokens, values) of a final-stream metric over the samples sweep."""
    ys, steps = get_ys(SAMPLES_SRC[model], model, (RANKME_HOOK, 'acts_centered'), yvar)
    return (None, None) if ys is None else (np.asarray(get_xs_tokens(model, steps), float),
                                           np.asarray(ys, float))

In [ ]:
# Animated engine (analysis/spectrum_anim.py): inject the data backend as the other anim
# notebooks do. The two panels here are 'scatter' and 'curve' — caller-supplied frames, so the
# scatter side needs no backend at all; the curve side resolves through get_ys above.
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
animate_spectra = sa.animate_spectra

In [ ]:
from IPython.display import display

def plane_frames(per_step, steps):
    """(rho, LogitVar, marker size) per checkpoint; size is the weight norm relative to the
    frame's largest, as in the static figure."""
    ki = k_index(per_step[steps[0]])
    return [(per_step[s]['rho'][ki].numpy(), per_step[s]['logitvar'].numpy(),
             1 + 30 * (per_step[s]['norm'] / per_step[s]['norm'].max()).numpy() ** 2)
            for s in steps]

def axes_coords(rho, logitvar, xlim, ylim):
    """The plane as drawn — position in axes fractions, LogitVar in decades."""
    lo, hi = np.log10(ylim)
    return ((rho - xlim[0]) / (xlim[1] - xlim[0]),
            (np.log10(np.where(logitvar > 0, logitvar, np.nan)) - lo) / (hi - lo))

def flow_frames(per_step, steps, xlim, ylim, nx=20, ny=16, radius=0.12, min_count=0.3,
                arrow=0.15, min_arrow=0.25, speed_gamma=0.3, mass_gamma=0.5):
    """Weight-norm-weighted field of where the cloud moves next (checkpoint t -> t+1).

    Each neuron carries its displacement in the drawn plane and a weight w = |w_out| over the
    frame's largest (the same weight the dot size uses). A grid point's arrow is the
    w-weighted mean displacement of the neurons within `radius` of it, times its share of
    weight-norm mass. The kernel is biweight — smooth (its slope vanishes at the edge), so a
    neuron drifting past a grid line moves the field continuously and nothing is binned, but
    compactly supported, so the field is drawn only where neurons actually are. A Gaussian
    was tried first: its tails let the dense core paint arrows across empty parts of the
    plane, because a few thousand neurons at 4 sigma still outweigh a handful nearby. Cells
    holding less than `min_count` neurons' worth of kernel weight are dropped. The last frame
    has no successor and is empty.

    Two exponents keep that readable, both < 1 for the same reason — the raw quantities span
    2-3 orders of magnitude, so a linear scale renders everything but the extreme as nothing:
      mass_gamma  the mass share is taken to this power, so the core keeps the long arrows
                  without the thin regions falling to nothing. Visibility down there is
                  `min_arrow`'s job — it floors the length at a fraction of the frame's
                  longest, so the high-rho tail, where the entropy neurons end up, reads at
                  all — which leaves this exponent free to set the ordering, not the floor.
      speed_gamma lengths are normalised per frame (longest = `arrow` of the box) and damped
                  by how slow the frame is against the fastest. The checkpoint schedule is
                  logarithmic, so the fastest frame moves ~30x further than the rest; at 1
                  (strictly comparable lengths) it flattens every other frame to nothing.
    """
    g = np.stack(np.meshgrid((np.arange(nx) + 0.5) / nx, (np.arange(ny) + 0.5) / ny), -1).reshape(-1, 2)
    ki = k_index(per_step[steps[0]])
    P = [axes_coords(per_step[s]['rho'][ki].numpy(), per_step[s]['logitvar'].numpy(), xlim, ylim)
         for s in steps]
    mass, count, disp = [], [], []
    for t in range(len(steps) - 1):
        (x0, y0), (x1, y1) = P[t], P[t + 1]
        ok = np.isfinite(x0) & np.isfinite(y0) & np.isfinite(x1) & np.isfinite(y1)
        w = (per_step[steps[t]]['norm'] / per_step[steps[t]]['norm'].max()).numpy()[ok]
        d2 = ((g[:, :1] - x0[ok]) ** 2 + (g[:, 1:] - y0[ok]) ** 2) / radius ** 2
        K = np.maximum(0.0, 1 - d2) ** 2
        kw = w * K
        mass.append(kw.sum(1)); count.append(K.sum(1))
        disp.append((kw @ (x1 - x0)[ok], kw @ (y1 - y0)[ok]))
    Z = max(m.max() for m in mass)
    scale = [np.where(m > 0, (m / Z) ** mass_gamma / np.where(m > 0, m, 1), 0) for m in mass]
    uv = [(u * s, v * s) for (u, v), s in zip(disp, scale)]   # local mean motion x mass share
    speed = np.array([np.hypot(u, v).max() for u, v in uv])
    gain = arrow * (speed / speed.max()) ** speed_gamma / np.where(speed > 0, speed, 1)
    out = []
    for n, (u, v), c, sp in zip(count, uv, gain, speed):
        k = n > min_count                                    # no neurons here -> no arrow
        u, v = c * u[k], c * v[k]
        L = np.hypot(u, v)
        floor = min_arrow * arrow * (sp / speed.max()) ** speed_gamma
        f = np.where(L > 0, np.maximum(L, floor) / np.where(L > 0, L, 1), 0)
        out.append((g[k, 0], g[k, 1], u * f, v * f))
    return out + [(np.zeros(0),) * 4]

def anim_entropy_plane(model, save_dir=None, fps=4, block=None, figsize=(13.5, 5.2), flow=False):
    b = last_block(model) if block is None else block
    per_step = data[model][b]
    steps = live_steps(per_step)
    toks = [float(per_step[s]['tokens']) for s in steps]
    frames = plane_frames(per_step, steps)
    lv = np.concatenate([f[1] for f in frames])
    xlim, ylim = (0.0, 1.0), sa._ylim(lv[lv > 0].min(), lv[lv > 0].max(), True)
    series = [(*stream_curve(model, y), c, lab, tw)
              for y, c, lab, tw in (('rankme', '0.35', 'RankMe', False),
                                    ('alpha', 'tab:purple', r'$\alpha$', True))]
    series = [s for s in series if s[1] is not None]
    panels = [('logitvar', None, [model], {
                  'kind': 'scatter', 'steps': steps, 'frames': frames,
                  'color': COLORS[model], 'xlim': xlim, 'ylim': ylim, 'xlog': False, 'ylog': True,
                  'field': flow_frames(per_step, steps, xlim, ylim) if flow else None,
                  'xlabel': r'$\rho$', 'ylabel': 'LogitVar',
                  'title': f'identification plane — blk{b} MLP'
                           + (' (+ weighted flow)' if flow else '')}),
              ('stream', None, [model], {
                  'kind': 'curve', 'series': series, 'scan_x': toks,
                  'xlabel': 'tokens', 'ylabel': 'RankMe', 'twin_ylabel': r'$\alpha$',
                  'title': f'final stream ({RANKME_HOOK}, centered)'})]
    labels = [f'step {s}  ({t:.2e} tokens)' for s, t in zip(steps, toks)]
    tag = 'entropy_plane_flow' if flow else 'entropy_plane'
    save = f'{save_dir}/{tag}_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, ncols=2, fps=fps, figsize=figsize, model=model, save=save,
                            frame_labels=labels, prog_bar='bottom',
                            suptitle=f'Entropy-neuron plane vs final-stream spectrum — '
                                     f'{get_model_label(model)}'))

## pythia-160m-deduped

In [ ]:
anim_entropy_plane('pythia-160m-deduped', save_dir='analysis/figures/animations')

## pythia-410m-deduped

In [ ]:
anim_entropy_plane('pythia-410m-deduped', save_dir='analysis/figures/animations')

## pythia-1b-deduped

In [ ]:
anim_entropy_plane('pythia-1b-deduped', save_dir='analysis/figures/animations')

## pythia-6.9b-deduped

In [ ]:
anim_entropy_plane('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

## nanochat-d12

In [ ]:
anim_entropy_plane('nanochat-d12', save_dir='analysis/figures/animations')

## OLMo-2-0425-1B

In [ ]:
anim_entropy_plane('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

## OLMo-2-1124-7B

In [ ]:
anim_entropy_plane('OLMo-2-1124-7B', save_dir='analysis/figures/animations')

# Weighted flow of the plane

Same animation with the cloud's motion overlaid: for every neuron, its step from this checkpoint to the next, spread over a grid by a kernel and weighted by its output weight norm, so heavy neurons dominate the local direction. Each arrow is that local weighted mean step times the share of weight-norm mass sitting under it, so a light neuron wandering alone barely registers. Nothing is binned: the kernel is smooth in space (its slope vanishes at the edge), so a neuron drifting past a grid line moves the field continuously. It is also compactly supported, so arrows appear wherever there are dots and nowhere else — a Gaussian was tried first and its tails let the dense core paint arrows across empty parts of the plane, since a few thousand neurons at 4σ outweigh a handful nearby.

Both scalings are compressed, because both quantities span orders of magnitude and a linear scale renders everything outside the core as invisible specks:

- **mass share** enters as $(\text{mass}/\text{mass}_{\max})^{0.5}$, and every drawn arrow is floored at 25% of the frame's longest. The floor is what keeps the sparse but interesting bottom-right legible; the exponent is then free to set the ordering among the regions that carry real weight.

- **frame speed**: lengths are normalised per frame (longest arrow = 15% of the box) and damped by that frame's speed against the fastest ($\text{speed}^{0.3}$). Strict cross-frame comparability was tried first and is unusable — the checkpoint schedule is logarithmic, so one early frame moves ~30x further than the rest and flattens every other frame to nothing. Across frames only the ordering of speeds survives.

The last checkpoint has no successor and shows no arrows.

## pythia-160m-deduped — flow

In [ ]:
anim_entropy_plane('pythia-160m-deduped', save_dir='analysis/figures/animations', flow=True)

## pythia-410m-deduped — flow

In [ ]:
anim_entropy_plane('pythia-410m-deduped', save_dir='analysis/figures/animations', flow=True)

## pythia-1b-deduped — flow

In [ ]:
anim_entropy_plane('pythia-1b-deduped', save_dir='analysis/figures/animations', flow=True)

## pythia-6.9b-deduped — flow

In [ ]:
anim_entropy_plane('pythia-6.9b-deduped', save_dir='analysis/figures/animations', flow=True)

## nanochat-d12 — flow

In [ ]:
anim_entropy_plane('nanochat-d12', save_dir='analysis/figures/animations', flow=True)

## OLMo-2-0425-1B — flow

In [ ]:
anim_entropy_plane('OLMo-2-0425-1B', save_dir='analysis/figures/animations', flow=True)

## OLMo-2-1124-7B — flow

In [ ]:
anim_entropy_plane('OLMo-2-1124-7B', save_dir='analysis/figures/animations', flow=True)

# Write magnitude over training

A different cut of the same neurons: every neuron of the last block ranked by how much it
actually writes into the residual stream — RMS activation over a token sample times the norm of
its output direction — coloured by $\rho$. The static version (section 10 of
`analysis/entropy_evidence.ipynb`) shows only the final checkpoint, where the null-space writers
sit at the very top of the ranking; animating it asks whether they were always the largest
writers or climbed there.

Activation statistics come from `data/results/neuron_write_stats.pt` (per checkpoint, written by
`oneoff_scripts/entropy_neuron_activations.py`), intersected with the weight sweep's checkpoints.
Models the sweep has not reached print a skip line.

In [ ]:
# Per-neuron write statistics for the ranking animations below. The per-checkpoint sweep wins
# per model; the older single-checkpoint file only fills models the sweep has not reached, and
# those have no int block keys, so the builder skips them.
_load = lambda p: torch.load(p, weights_only=False) if os.path.exists(p) else {}
ws = {**_load('data/results/entropy_neuron_activations.pt'),
      **_load('data/results/neuron_write_stats.pt')}
print({m: sorted(b for b in v if isinstance(b, int)) for m, v in ws.items()})

In [ ]:
def write_rank_frames(model, block=None):
    """(rank, write magnitude, marker size, rho) per checkpoint, sorted by magnitude.

    Checkpoints whose write matrix is still zero (nanochat's zero-init) are dropped: their
    magnitudes are all zero and the log axis has nothing to draw.
    """
    b = max(k for k in ws[model] if isinstance(k, int)) if block is None else block
    frames, steps = [], []
    for s in sorted(set(ws[model][b]) & set(data[model][b])):
        q = data[model][b][s]
        a_, w_ = ws[model][b][s]['rms'].numpy(), q['norm'].numpy()
        r_ = q['rho'][k_index(q)].numpy()
        n = min(len(a_), len(w_), len(r_))
        mag = a_[:n] * w_[:n]
        if not mag.max() > 0:
            continue
        o = np.argsort(-mag)
        frames.append((np.arange(n), mag[o], np.full(n, 4.0), r_[:n][o]))
        steps.append(s)
    return frames, steps

def anim_write_rank(model, save_dir=None, fps=4, figsize=(7.6, 5.2)):
    if model not in ws or model not in data or not any(isinstance(k, int) for k in ws[model]):
        return print(f'[skipped] no per-checkpoint activation sweep for {model}')
    frames, steps = write_rank_frames(model)
    b = max(k for k in ws[model] if isinstance(k, int))
    ntk = int(ws[model][b][steps[0]]['n'])
    panels = [('write_magnitude', None, [model], {
                  'kind': 'scatter', 'steps': steps, 'frames': frames,
                  'xlog': False, 'ylog': True, 'alpha': 1.0,
                  'cmap': 'viridis', 'vmin': 0, 'vmax': 1, 'cbar_label': r'$\rho$',
                  'xlabel': 'neuron rank by write magnitude',
                  'ylabel': r'RMS($a_i$)$\cdot\|w_{out}\|$',
                  'title': f'{get_model_label(model)} — blk{b}, '
                           f'{ntk / 1000:.0f}k tokens/checkpoint'})]
    labels = [f'step {s}  ({get_xs_tokens(model, [s])[0]:.2e} tokens)' for s in steps]
    save = f'{save_dir}/write_rank_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, ncols=1, fps=fps, figsize=figsize, model=model, save=save,
                            frame_labels=labels, prog_bar='top'))

## pythia-160m-deduped — write magnitude

In [ ]:
anim_write_rank('pythia-160m-deduped', save_dir='analysis/figures/animations')

## pythia-410m-deduped — write magnitude

In [ ]:
anim_write_rank('pythia-410m-deduped', save_dir='analysis/figures/animations')

## pythia-1b-deduped — write magnitude

In [ ]:
anim_write_rank('pythia-1b-deduped', save_dir='analysis/figures/animations')

## pythia-6.9b-deduped — write magnitude

In [ ]:
anim_write_rank('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

## nanochat-d12 — write magnitude

In [ ]:
anim_write_rank('nanochat-d12', save_dir='analysis/figures/animations')

## OLMo-2-0425-1B — write magnitude

In [ ]:
anim_write_rank('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

## OLMo-2-1124-7B — write magnitude

In [ ]:
anim_write_rank('OLMo-2-1124-7B', save_dir='analysis/figures/animations')